# HW1.1: Dictionary-based Tokenization


In this exercise, you are to implement a dictionary-based word segmentation algorithm. There are two Python functions that you need to complete:
<br>
* maximal_matching
* backtrack
</br>

Also, you have to find how to use word_tokenize() in PythaiNLP along with customer_dict by yourselves.

In [41]:
!pip install pythainlp
!pip install marisa_trie
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import get_corpus
from marisa_trie import Trie

## Part 1) Maximal Matching from PythaiNLP

### Create a toy dictionary to test the algorithm

This is based on the example shown in the lecture.
You will tokenize the following text string: "ไปหามเหสี!"
The toy dictionary provided in this exercise includes all the charaters, syllables, and words that appear that the text string.

In [42]:
thai_vocab = ["ไ","ป","ห","า","ม","เ","ห","ส","ี","ไป","หา","หาม","เห","สี","มเหสี","!"]
input_text = "ไปหามเหสี!"

### Example Dictionary

Write the `word_tokenize` function of PyThaiNLP with a custom dictionary above and using:
1. Longest matching algorithm `longest`
2. Maximal-matching algorithm `newmm`

Study `word_tokenize()` from PythaiNLP in the link below. Note: `custom_dict` will accept Trie structures as `Trie(iterable)`.

https://pythainlp.org/docs/5.0/api/tokenize.html#pythainlp.tokenize.word_tokenize

In [43]:
####FILL CODE HERE####
from pythainlp.tokenize import word_tokenize
from pythainlp.util import Trie

# Custom dictionary initialization
custom_dict = Trie(thai_vocab)

tokens_longest = word_tokenize(
    text=input_text,
    engine="longest",
    custom_dict=custom_dict
)

tokens_newmm = word_tokenize(
    text=input_text,
    engine="newmm",
    custom_dict=custom_dict
)

print(tokens_longest)

print(tokens_newmm)

######################

['ไป', 'หาม', 'เห', 'สี', '!']
['ไป', 'หา', 'มเหสี', '!']


## Part 2) Maximal Matching from Scratch

### Maximal matching
Complete the maximal matching function below with dynamic programming to tokenize the input text and output the 2D numerical array shown in class.

In [44]:
from math import inf #infinity
def maximal_matching(c):
    #Initialize an empty 2D list
    d  =[[None]*len(c) for _ in range(len(c))]

    ####FILL CODE HERE####
# Build the table column by column (end index j)
    for j in range(len(c)):
        # For each column j, consider substrings starting at i from 0..j
        for i in range(j+1):
            # Extract the substring c[i..j+1]
            sub = c[i:j+1]

            if sub in thai_vocab:
                # If the substring starts at 0, it's just 1 token
                if i == 0:
                    d[i][j] = 1
                else:
                    # We add 1 token to the best cost up to index i-1
                    best_left = inf
                    for k in range(i):
                        if d[k][i-1] is not None and d[k][i-1] != inf:
                            if d[k][i-1] < best_left:
                                best_left = d[k][i-1]

                    if best_left == inf:
                        d[i][j] = inf
                    else:
                        d[i][j] = best_left + 1
            else:
                # Not in vocab => can't directly tokenize => inf
                d[i][j] = inf
    ######################

    return d

### Backtracking
Complete the backtracking function below to find the tokenzied words.
It should return a list containing a pair of the beginning position and the ending position of each word.
In this example, it should return:
<br>
[(0, 1),(2, 3),(4, 8),(9, 9)]
<br>
#### Each pair contains the position of each word as follows:
(0, 1) ไป
<br>
(2, 3) หา
<br>
(4, 8) มเหสี
<br>
(9, 9) !


In [45]:
def backtrack(d):
    eow = len(d)-1 # End of Word position
    word_pos = [] # Word position
    ####FILL CODE HERE####
    while eow >= 0:
        best_cost = float('inf')
        best_i = None
        # Find the start index i that gives minimal cost in d[i][eow]
        for i in range(eow + 1):
            cost = d[i][eow]
            if cost is not None and cost != float('inf') and cost < best_cost:
                best_cost = cost
                best_i = i

        # best_i now points to the start of the token ending at eow
        word_pos.append((best_i, eow))
        eow = best_i - 1  # Jump left to continue backtracking
    ######################
    word_pos.reverse()
    return word_pos

### Test your maximal matching algorithm on a toy dictionary

Expected output:

[1, 1, inf, inf, inf, inf, inf, inf, inf, inf] ไ
<br>
[None, 2, inf, inf, inf, inf, inf, inf, inf, inf] ป
<br>
[None, None, 2, 2, 2, inf, inf, inf, inf, inf] ห
<br>
[None, None, None, 3, inf, inf, inf, inf, inf, inf] า
<br>
[None, None, None, None, 3, inf, inf, inf, 3, inf] ม
<br>
[None, None, None, None, None, 3, 3, inf, inf, inf] เ
<br>
[None, None, None, None, None, None, 4, inf, inf, inf] ห
<br>
[None, None, None, None, None, None, None, 4, 4, inf] ส
<br>
[None, None, None, None, None, None, None, None, 5, inf] ี
<br>
[None, None, None, None, None, None, None, None, None, 4] !
<br>

In [46]:
input_text = "ไปหามเหสี!"
out = maximal_matching(input_text)
for i in range(len(out)):
    print(out[i],input_text[i])

[1, 1, inf, inf, inf, inf, inf, inf, inf, inf] ไ
[None, 2, inf, inf, inf, inf, inf, inf, inf, inf] ป
[None, None, 2, 2, 2, inf, inf, inf, inf, inf] ห
[None, None, None, 3, inf, inf, inf, inf, inf, inf] า
[None, None, None, None, 3, inf, inf, inf, 3, inf] ม
[None, None, None, None, None, 3, 3, inf, inf, inf] เ
[None, None, None, None, None, None, 4, inf, inf, inf] ห
[None, None, None, None, None, None, None, 4, 4, inf] ส
[None, None, None, None, None, None, None, None, 5, inf] ี
[None, None, None, None, None, None, None, None, None, 4] !


### Test your backtracking algorithm on a toy dictionary
Compare your results with the result from PyThaiNLP `newmm`.

Expected output:
<br>
ไป|หา|มเหสี|!

In [47]:
def print_tokenized_text(d, input_text):
    tokenized_text=[]
    for pos in backtrack(d):
        #print(pos)
        tokenized_text.append(input_text[pos[0]:pos[1]+1])

    print("|".join(tokenized_text))

print_tokenized_text(out,input_text)

ไป|หา|มเหสี|!


### <font color=blue>Question 1</font>
Using your maximal matching code with the toy dictionary, how many “words” did you get when tokenizing this input text.

Answer this question in question #1 in MyCourseVille. Also print out the answer in this notebook as well.

In [48]:
input_text = "ไปหาหมามเหสีมาหาม!"

In [52]:
out = maximal_matching(input_text)
for i in range(len(out)):
    print(out[i],input_text[i])

print_tokenized_text(out,input_text)

[1, 1, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf] ไ
[None, 2, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf] ป
[None, None, 2, 2, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf] ห
[None, None, None, 3, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf] า
[None, None, None, None, 3, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf] ห
[None, None, None, None, None, 4, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf] ม
[None, None, None, None, None, None, 5, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf] า
[None, None, None, None, None, None, None, 6, inf, inf, inf, 6, inf, inf, inf, inf, inf, inf] ม
[None, None, None, None, None, None, None, None, 7, 7, inf, inf, inf, inf, inf, inf, inf, inf] เ
[None, None, None, None, None, None, None, None, None, 8, inf, inf, inf, inf, inf, inf, inf, inf] ห
[None, None, None, None, None, None, None, None, None

## Part 3) Your Maximal Matching with Real Dictionary

For UNIX-based OS users, the following cell will download a dictionary (it's just a list of thai words). Alternatively, you can download it from this link: https://raw.githubusercontent.com/PyThaiNLP/pythainlp/dev/pythainlp/corpus/words_th.txt

In [53]:
!wget https://raw.githubusercontent.com/PyThaiNLP/pythainlp/dev/pythainlp/corpus/words_th.txt

--2025-02-19 14:13:07--  https://raw.githubusercontent.com/PyThaiNLP/pythainlp/dev/pythainlp/corpus/words_th.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1519661 (1.4M) [text/plain]
Saving to: ‘words_th.txt.2’

words_th.txt.2      100%[===================>]   1.45M  --.-KB/s    in 0.01s   

2025-02-19 14:13:08 (123 MB/s) - ‘words_th.txt.2’ saved [1519661/1519661]



In [54]:
with open("words_th.txt",encoding='utf-8-sig') as f:
    thai_vocab = f.read().splitlines()
print("Vocab size:", len(thai_vocab))
print(thai_vocab[:10])

thai_vocab.extend(["ๆ","!"])

Vocab size: 62080
['ก ข ไม่กระดิกหู', 'ก.', 'ก.ค.', 'ก.ต.', 'ก.ป.ส.', 'ก.พ.', 'ก.พ.ด.', 'ก.ม.', 'ก.ย.', 'ก.ย']


### Part 3.1) The output of **YOUR** maximal matching algoithm on the new dictionary
Expected output:
<br>
[inf, 1, inf, 1, inf, inf, inf, inf, inf] ไ
<br>
[None, inf, inf, inf, inf, inf, inf, inf, inf] ป
<br>
[None, None, inf, 2, 2, inf, inf, inf, inf] ห
<br>
[None, None, None, inf, inf, inf, inf, inf, inf] า
<br>
[None, None, None, None, inf, inf, inf, inf, 2] ม
<br>
[None, None, None, None, None, inf, 3, inf, inf] เ
<br>
[None, None, None, None, None, None, inf, inf, inf] ห
<br>
[None, None, None, None, None, None, None, inf, 4] ส
<br>
[None, None, None, None, None, None, None, None, inf] ี

### Expected tokenized text
ไปหา|มเหสี

_Question: Why are the resulting tokens different?_

In [55]:
input_text = "ไปหามเหสี"
out = maximal_matching(input_text)
for i in range(len(out)):
    print(out[i],input_text[i])

print_tokenized_text(out,input_text)

[inf, 1, inf, 1, inf, inf, inf, inf, inf] ไ
[None, inf, inf, inf, inf, inf, inf, inf, inf] ป
[None, None, inf, 2, 2, inf, inf, inf, inf] ห
[None, None, None, inf, inf, inf, inf, inf, inf] า
[None, None, None, None, inf, inf, inf, inf, 2] ม
[None, None, None, None, None, inf, 3, inf, inf] เ
[None, None, None, None, None, None, inf, inf, inf] ห
[None, None, None, None, None, None, None, inf, 4] ส
[None, None, None, None, None, None, None, None, inf] ี
ไปหา|มเหสี


### <font color=blue>Question 2</font>
Using your maximal matching algorithm and the actual Thai dictionary, how many “words” did you get when tokenizing this input text.

Answer this question in question #2 in MyCourseVille. Also print out the answer in this notebook as well.

In [62]:
input_text = "ประเทศไทยรวมเลือดเนื้อชาติเชื้อไทยเป็นประชารัฐไผทของไทยทุกส่วนอยู่ดำรงคงไว้ได้ทั้งมวลด้วยไทยล้วนหมายรักสามัคคี"
# Compute DP table
out = maximal_matching(input_text)

# Show final tokenization
print_tokenized_text(out, input_text)

ประเทศ|ไทย|รวม|เลือดเนื้อ|ชาติ|เชื้อ|ไทย|เป็น|ประชา|รัฐ|ไผท|ของ|ไทย|ทุก|ส่วน|อยู่|ดำรง|คงไว้|ได้|ทั้งมวล|ด้วย|ไทย|ล้วน|หมาย|รัก|สามัคคี


### Part 3.2) Use PyThaiNLP `word_tokenize` with custom dictionary

Try tokenizing the following text with `word_tokenize` in `newmm` algorithm and default real dictionary.

In [57]:
text='นัดกินกันตอนไหนก็ได้ที่สามย่านมิตรทาวน์'

####FILL CODE HERE####
tokens_newmm = word_tokenize(text, engine="newmm")

print(tokens_newmm)
######################

['นัด', 'กินกัน', 'ตอน', 'ไหน', 'ก็', 'ได้ที่', 'สามย่าน', 'มิตร', 'ทาวน์']


Add 'สามย่านมิตรทาวน์' into dictionary and then tokenize again

In [58]:
####FILL CODE HERE####
from pythainlp.corpus.common import thai_words  # Default Thai word list
text = "นัดกินกันตอนไหนก็ได้ที่สามย่านมิตรทาวน์"

vocab = set(thai_words())
vocab.add("สามย่านมิตรทาวน์")

custom_dict = Trie(list(vocab))

tokens_custom = word_tokenize(text, engine="newmm", custom_dict=custom_dict)
print(tokens_custom)
######################

['นัด', 'กินกัน', 'ตอน', 'ไหน', 'ก็', 'ได้ที่', 'สามย่านมิตรทาวน์']


### <font color=blue>Question 3</font>
Using the code from part three only, how many “words” did you get when tokenizing this input text **after adding the new vocabs**.

Answer this question in question #3 in MyCourseVille. Also print out the answer in this notebook as well.

In [66]:
new_vocab = ["ดิสนีย์ออนไอซ์", "ตีกอล์ฟ", "ธรรมมะ"]
input_text = "อ๋อก็ว่าจะไปเรียนแต่งหน้านั่งสมาธิดำน้ำปลูกปะการังทำอาหารนวดสปาปลูกป่าดำนาดูดิสนีย์ออนไอซ์แรลลี่ตีกอล์ฟล่องเรือส่องสัตว์ช้อปปิ้งดูงิ้วดูละครเวทีดูคอนเสิร์ตดินเนอร์ทำขนมจัดดอกไม้เที่ยวตลาดน้ำเรียนถ่ายรูปดูกายกรรมชมเมืองเก่าเข้าสัมมนาทัวร์ธรรมมะเรียนเต้นแล้วก็ร้องเพลง"

default_vocab = set(thai_words())

for w in new_vocab:
    default_vocab.add(w)

custom_trie = Trie(list(default_vocab))

tokens_custom = word_tokenize(
    input_text,
    engine="newmm",
    custom_dict=custom_trie
)

print(tokens_custom)
print("Number of tokens:", len(tokens_custom))

['อ๋อ', 'ก็', 'ว่า', 'จะ', 'ไป', 'เรียน', 'แต่งหน้า', 'นั่งสมาธิ', 'ดำน้ำ', 'ปลูก', 'ปะการัง', 'ทำอาหาร', 'นวด', 'สปา', 'ปลูกป่า', 'ดำนา', 'ดู', 'ดิสนีย์ออนไอซ์', 'แรลลี่', 'ตีกอล์ฟ', 'ล่องเรือ', 'ส่องสัตว์', 'ช้อปปิ้ง', 'ดู', 'งิ้ว', 'ดู', 'ละครเวที', 'ดู', 'คอนเสิร์ต', 'ดินเนอร์', 'ทำ', 'ขนม', 'จัด', 'ดอกไม้', 'เที่ยว', 'ตลาดน้ำ', 'เรียน', 'ถ่ายรูป', 'ดู', 'กายกรรม', 'ชม', 'เมือง', 'เก่า', 'เข้า', 'สัมมนา', 'ทัวร์', 'ธรรมมะ', 'เรียน', 'เต้น', 'แล้วก็', 'ร้องเพลง']
Number of tokens: 51


## Part 4) Use maximal matching on real dataset

To complete this exercise, we will use the maximal matching algorithm on NECTEC's BEST corpus.

The corpus has a structure of characters with target whether it's a beginning of a word (True/False).

In [67]:
#Download dataset
!gdown "1EcrlXYUyIEM3aeIJse6nPpiv_UjSKgoU&confirm=t"

Downloading...
From: https://drive.google.com/uc?id=1EcrlXYUyIEM3aeIJse6nPpiv_UjSKgoU&confirm=t
To: /content/corpora.tar.gz
100% 121M/121M [00:00<00:00, 140MB/s]


In [68]:
!tar xvf corpora.tar.gz

corpora/
corpora/mnist_data/
corpora/mnist_data/t10k-images-idx3-ubyte.gz
corpora/mnist_data/train-images-idx3-ubyte.gz
corpora/mnist_data/.ipynb_checkpoints/
corpora/mnist_data/vis_utils.py
corpora/mnist_data/__init__.py
corpora/mnist_data/load_mnist.py
corpora/mnist_data/train-labels-idx1-ubyte.gz
corpora/mnist_data/t10k-labels-idx1-ubyte.gz
corpora/BEST/
corpora/BEST/test/
corpora/BEST/test/df_best_article_test.csv
corpora/BEST/test/df_best_encyclopedia_test.csv
corpora/BEST/test/df_best_novel_test.csv
corpora/BEST/test/df_best_news_test.csv
corpora/BEST/train/
corpora/BEST/train/df_best_encyclopedia_train.csv
corpora/BEST/train/df_best_article_train.csv
corpora/BEST/train/df_best_news_train.csv
corpora/BEST/train/df_best_novel_train.csv
corpora/BEST/val/
corpora/BEST/val/df_best_encyclopedia_val.csv
corpora/BEST/val/df_best_news_val.csv
corpora/BEST/val/df_best_article_val.csv
corpora/BEST/val/df_best_novel_val.csv
corpora/.ipynb_checkpoints/
corpora/.ipynb_checkpoints/Word_Tokeniz

In [19]:
import pandas as pd
import os

In [69]:
# Path to the preprocessed data
best_processed_path = 'corpora/BEST'
option = "test"

df = []
# article types in BEST corpus
article_types = ['article', 'encyclopedia', 'news', 'novel']
for article_type in article_types:
    df.append(pd.read_csv(os.path.join(best_processed_path, option, 'df_best_{}_{}.csv'.format(article_type, option))))
df = pd.concat(df)
df

,char,target
0,ป,True
1,ฏ,False
2,ิ,False
3,ร,False
4,ู,False
...,...,...
644911,ห,False
644912,น,False
644913,ม,True
644914,า,False


In [70]:
len(df)

2271932

In [71]:
# Some text in this corpus
all_text = "".join(df['char'].tolist())
all_text[:1000]

'ปฏิรูปการศึกษา : มุมมองทางกระบวนทัศน์และบริบทสังคมไทยThe Reformation of Eucation from A Thai Perspectiveกระบวนทัศน์และวิธีคิดแบบแยกส่วน ลดส่วน ได้ทำให้"การศึกษาเรียนรู้"ใน หลายทศวรรษที่ผ่านมา กลายเป็นเรื่องของนักวิชาการด้านศึกษาศาสตร์ ครุศาสตร์ หรือเป็นเรื่องของโรงเรียน ครูอาจารย์ กระทรวงศึกษาธิการ ทบวงมหาวิทยาลัยฯ มาอย่างต่อเนื่องยาวนาน (เหมือนกับที่เรื่องสุขภาพเป็นเรื่องของแพทย์และโรงพยาบาล) การจัดการศึกษาภายใต้กระบวนทัศน์และวิธีคิดแบบดังกล่าวของรัฐ ได้ถูกวิพากษ์วิจารณ์และตกเป็นจำเลยจากวิกฤตการณ์ทางสังคมมากมาย อันสะท้อนถึงความล้มเหลวของการจัดการศึกษาเพื่อพัฒนามนุษย์ (ปัญหาศีลธรรมเสื่อมถอย ยาเสพติด การขาดจิตสำนึกทางสังคม ฯลฯ) ซึ่งสังคมร่วมกันสรุปว่า เกิดจากความล้มเหลวของระบบการศึกษาในกระบวนทัศน์แบบแยกส่วน นำมาสู่การปฏิรูปการศึกษาที่กำลังดำเนินการอยู่ในปัจจุบัน ด้วยเป้าหมายเพื่อสร้างการเรียนรู้แบบองค์รวม ที่จะทำให้"ผู้เรียนเก่ง-ดี-มีความสุข"คำถามที่ผู้เขียนสนใจในการปฏิรูปการศึกษาที่ดำเนินการในปัจจุบัน คือ๑.การปฏิรูปการศึกษาในปัจจุบันดำเนินการภายใต้กระบวนทัศน์แบบบูรณาการ (องค์รวม)ตามที

### <font color=blue>Question 4</font>
Using PyThaiNLP `newmm`, how many words did you get in the BEST corpus (test)? [Runtime is around 7 mins] What are the accuracy, f1, precision, recall scores for each character?

Answer this question in question #4 in MyCourseVille. Also print out the answer in this notebook as well.

_Question: What main metric should we look at? Why?_

In [72]:
####FILL CODE HERE####
y_true = df["target"].astype(int).values  # convert True->1, False->0

# 3) Reconstruct the text
all_text = "".join(df["char"].tolist())

# 4) Tokenize all_text with newmm
tokens = word_tokenize(all_text, engine="newmm")

print("Number of tokens (newmm):", len(tokens))
######################

Number of tokens (newmm): 569631


In [73]:
def convert_to_character(_tokens):
  char_list = [0]*len("".join(_tokens))
  char_count = 0
  for word in _tokens:
    char_list[char_count] = 1
    char_count += len(word)
  return char_list

chars = convert_to_character(tokens)
chars[:20]

[1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0]

In [74]:
from sklearn.metrics import f1_score,precision_score,recall_score,accuracy_score

####FILL CODE HERE####
acc = accuracy_score(y_true, chars)
prec = precision_score(y_true, chars)
rec = recall_score(y_true, chars)
f1 = f1_score(y_true, chars)

print("Number of tokens predicted by newmm:", len(tokens))
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)

# 7) Answer which metric to focus on
print("Main Metric to look at: F1, because of class imbalance and the need to balance Precision and Recall.")
######################

Number of tokens predicted by newmm: 569631
Accuracy: 0.9431373826329309
Precision: 0.9422661336900555
Recall: 0.8478765332638281
F1: 0.8925828735253718
Main Metric to look at: F1, because of class imbalance and the need to balance Precision and Recall.
